In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import vstack
from collections import Counter

#Load news dataset with the columns
news_df = pd.read_csv(
    "../MINDsmall_train/news.tsv",
    sep="\t",
    header=None,
    names=[
        "news_id", "category", "subcategory",
        "title", "abstract", "url",
        "title_entities", "abstract_entities"
    ]
)

#Preprocess the data to fill missing values 
news_df["title"]    = news_df["title"].fillna("")
news_df["abstract"] = news_df["abstract"].fillna("")
news_df["url"]      = news_df["url"].fillna("")

#Select required columns
news_df = news_df[["news_id", "category", "subcategory", "title", "abstract", "url"]]

#Combine title and abstract column
news_df["content"] = news_df["title"] + " " + news_df["abstract"]

#Load behaviors with the columns
behaviors_df = pd.read_csv(
    "../MINDsmall_train/behaviors.tsv",
    sep="\t",
    header=None,
    names=["impression_id", "user_id", "time", "history", "impressions"]
)

#Preprocess the data to fill missing values and split history into a list

behaviors_df["history"] = behaviors_df["history"].fillna("")
behaviors_df["history"] = behaviors_df["history"].apply(lambda x: x.split())

#Create a TD-IDF matrix for the news content
tfidf = TfidfVectorizer(stop_words="english", max_features=5000)
tfidf_matrix = tfidf.fit_transform(news_df["content"])

#Map NEWS_ID
news_id_to_index = {nid: idx for idx, nid in enumerate(news_df["news_id"])}

#Recency score
news_df["recency_score"] = news_df.index / len(news_df)

print(f"News loaded       : {len(news_df):,} articles")
print(f"Behaviors loaded  : {len(behaviors_df):,} records")
print(f"TF-IDF matrix     : {tfidf_matrix.shape}")

News loaded       : 51,282 articles
Behaviors loaded  : 156,965 records
TF-IDF matrix     : (51282, 5000)


In [4]:
#Existing functions

def build_user_profile(clicked_news_ids):
    """
    Weighted average of TF-IDF vectors for clicked articles.
    More recent clicks receive higher weight: weight = (i+1) / total
    """
    vectors = []
    weights = []
    total   = len(clicked_news_ids)

    for i, news_id in enumerate(clicked_news_ids):
        if news_id in news_id_to_index:
            idx = news_id_to_index[news_id]
            vectors.append(tfidf_matrix[idx])
            weights.append((i + 1) / total)

    if len(vectors) == 0:
        return None

    stacked          = vstack(vectors)
    weights          = np.array(weights).reshape(-1, 1)
    weighted_profile = stacked.multiply(weights).sum(axis=0) / weights.sum()

    return np.asarray(weighted_profile)


def recommend_articles_with_recency(
    user_profile,
    clicked_news_ids,
    selected_category=None,
    selected_subcategory=None,
    alpha=0.9,
    beta=0.1,
    top_n=5,
    candidate_pool=100
):
    """
    Rank articles by: final_score = alpha * similarity + beta * recency
    Scans top `candidate_pool` similarity results, then re-sorts by final_score.
    Filters out clicked articles, and optionally by category / subcategory.
    """
    similarity_scores = cosine_similarity(user_profile, tfidf_matrix).flatten()
    sorted_indices    = similarity_scores.argsort()[::-1]

    recommendations = []

    for idx in sorted_indices:
        row     = news_df.iloc[idx]
        news_id = row["news_id"]

        if news_id in clicked_news_ids:
            continue
        if selected_category    and row["category"]    != selected_category:
            continue
        if selected_subcategory and row["subcategory"] != selected_subcategory:
            continue

        sim_score     = similarity_scores[idx]
        recency_score = row["recency_score"]
        final_score   = alpha * sim_score + beta * recency_score

        recommendations.append((idx, sim_score, recency_score, final_score))

        if len(recommendations) >= candidate_pool:
            break

    recommendations = sorted(recommendations, key=lambda x: x[3], reverse=True)[:top_n]

    results = []
    for idx, sim, rec, final in recommendations:
        row = news_df.iloc[idx][["news_id", "title", "abstract", "category", "subcategory", "url"]].to_dict()
        row["similarity_score"] = round(sim,   4)
        row["recency_score"]    = round(rec,   4)
        row["final_score"]      = round(final, 4)
        results.append(row)

    return pd.DataFrame(results)


def create_user_profile_from_interest(category, subcategory, n_seed=5):
    """
    Cold-start: builds an initial profile from seed articles
    in the chosen category/subcategory.
    """
    seed_articles = news_df[
        (news_df["category"]    == category) &
        (news_df["subcategory"] == subcategory)
    ].head(n_seed)

    clicked_ids  = seed_articles["news_id"].tolist()
    user_profile = build_user_profile(clicked_ids)

    return user_profile, clicked_ids


print("Core functions loaded.")

Core functions loaded.


In [5]:
#Popularity scores. Calculate how many times each articles was cliekced across all behaviours.

def compute_popularity_scores(behaviors_df, news_df):
    """
    Count how many times each article appears as a click (label=1)
    across all impressions in behaviors_df.
    Normalise to [0, 1].
    """
    click_counts = Counter()

    for impression_str in behaviors_df["impressions"]:
        # impressions column is still raw string here
        for item in impression_str.split():
            news_id, label = item.split("-")
            if int(label) == 1:
                click_counts[news_id] += 1

    # Map counts onto news_df
    news_df["popularity_raw"]   = news_df["news_id"].map(click_counts).fillna(0)
    max_clicks                  = news_df["popularity_raw"].max()
    news_df["popularity_score"] = news_df["popularity_raw"] / max_clicks if max_clicks > 0 else 0

    return news_df


# Load raw behaviors for popularity counting (impressions still as strings)
behaviors_raw = pd.read_csv(
    "../MINDsmall_train/behaviors.tsv",
    sep="\t",
    header=None,
    names=["impression_id", "user_id", "time", "history", "impressions"]
)

news_df = compute_popularity_scores(behaviors_raw, news_df)

print(f"Popularity scores computed.")
print(f"Most clicked article : {int(news_df['popularity_raw'].max())} clicks")
print(f"Articles with 0 clicks: {(news_df['popularity_raw'] == 0).sum():,}")
news_df[["news_id", "popularity_raw", "popularity_score"]].sort_values(
    "popularity_raw", ascending=False
).head(5)

Popularity scores computed.
Most clicked article : 4316 clicks
Articles with 0 clicks: 43,569


,news_id,popularity_raw,popularity_score
33899,N55689,4316,1.000000
50688,N35729,3346,0.775255
41550,N33619,3246,0.752085
45974,N53585,2835,0.656858
41552,N63970,2578,0.597312
